# Specialty Prioritisation Framework 
### Purpose:

Define framework
- Import indicator datasets
- Build specialty master datasets
- Run EDA
- Export outputs

Develop a reproducible framework to identify clinical specialties demonstrating strong signals of:
 
- Disease burden
- Poor outcomes
- System pressure
- Geographic variation
 
##### This framework is intended to support Horizon Scanning and Demand Signalling by providing a transparent evidence base for identifying themes that may warrant further investigation.

In [31]:
#import relevant libraries

import pandas as pd
import numpy as np

pd.set_option("display.max_rows", None)

In [32]:
#build framework

framework = pd.DataFrame({

    "Specialty": [

        "Cardiovascular",
        "Cardiovascular",
        "Cardiovascular",
        "Cardiovascular",
        "Cardiovascular",

        "Respiratory",
        "Respiratory",
        "Respiratory",
        "Respiratory",

        "Mental Health",
        "Mental Health",
        "Mental Health"

    ],

    "Domain": [

        "Outcome",
        "Pressure",
        "Burden",
        "Burden",
        "Burden",

        "Outcome",
        "Pressure",
        "Burden",
        "Burden",

        "Outcome",
        "Pressure",
        "Burden"

    ],

    "Indicator": [

        "Under75_CVD_Mortality",
        "Cardiology_RTT_18plus",
        "Heart_Disease_Prevalence",
        "Heart_Failure_Prevalence",
        "Hypertension_Prevalence",

        "Under75_Respiratory_Mortality",
        "Respiratory_RTT_18plus",
        "COPD_Prevalence",
        "Asthma_Prevalence",

        "SMI_Excess_Mortality",
        "Mental_Health_Wait",
        "SMI_Prevalence"

    ],

    "Source": [

        "MHS",
        "MHS",
        "MHS",
        "MHS",
        "MHS",

        "MHS",
        "MHS",
        "MHS",
        "MHS",

        "MHS",
        "MHS",
        "MHS"

    ],

    "Direction": [

        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse",

        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse",

        "Higher_Worse",
        "Higher_Worse",
        "Higher_Worse"

    ]
})

framework

,Specialty,Domain,Indicator,Source,Direction
0,Cardiovascular,Outcome,Under75_CVD_Mortality,MHS,Higher_Worse
1,Cardiovascular,Pressure,Cardiology_RTT_18plus,MHS,Higher_Worse
2,Cardiovascular,Burden,Heart_Disease_Prevalence,MHS,Higher_Worse
3,Cardiovascular,Burden,Heart_Failure_Prevalence,MHS,Higher_Worse
4,Cardiovascular,Burden,Hypertension_Prevalence,MHS,Higher_Worse
5,Respiratory,Outcome,Under75_Respiratory_Mortality,MHS,Higher_Worse
6,Respiratory,Pressure,Respiratory_RTT_18plus,MHS,Higher_Worse
7,Respiratory,Burden,COPD_Prevalence,MHS,Higher_Worse
8,Respiratory,Burden,Asthma_Prevalence,MHS,Higher_Worse
9,Mental Health,Outcome,SMI_Excess_Mortality,MHS,Higher_Worse


In [33]:
specialty_inventory = (
    framework
    .groupby("Specialty")["Indicator"]
    .count()
    .reset_index()
)

specialty_inventory.columns = [
    "Specialty",
    "Number_of_Indicators"
]

specialty_inventory

,Specialty,Number_of_Indicators
0,Cardiovascular,5
1,Mental Health,3
2,Respiratory,4


## Cardiovascular Master Dataset

In [34]:
def load_mhs_indicator(
    file_path,
    indicator_name
):

    df = pd.read_excel(file_path)

    df = df[
        [
            "OrganisationName",
            "OrganisationCode",
            "OrganisationRawValue"
        ]
    ]

    df.columns = [
        "ICB_Name",
        "ICB_Code",
        indicator_name
    ]

    return df

#### Load cardiovascular indicators

In [35]:
cvd_mortality = load_mhs_indicator(
    r"..\Inputs\Under75_CVD_Mortality.xlsx",
    "Under75_CVD_Mortality"
)

cardiology_rtt = load_mhs_indicator(
    r"..\Inputs\Cardiology_RTT_18plus.xlsx",
    "Cardiology_RTT_18plus"
)

coronary_heart_disease = load_mhs_indicator(
    r"..\Inputs\Coronary_Heart_Disease_Prevalence.xlsx",
    "Coronary_Heart_Disease_Prevalence"
)

heart_failure = load_mhs_indicator(
    r"..\Inputs\Heart_Failure_PrevalencePercentage.xlsx",
    "Heart_Failure_PrevalencePercentage"
)

hypertension = load_mhs_indicator(
    r"..\Inputs\Hypertension_Prevalence.xlsx",
    "Hypertension_Prevalence"
)

##### Check every raw indicator immediately after loading

In [36]:
indicator_datasets = {
    "CVD mortality": cvd_mortality,
    "Cardiology RTT": cardiology_rtt,
    "Coronary heart disease prevalence": coronary_heart_disease,
    "Heart failure prevalence": heart_failure,
    "Hypertension prevalence": hypertension,
}

coverage_summary = []

for indicator_name, df in indicator_datasets.items():

    coverage_summary.append({
        "Indicator": indicator_name,
        "Rows": len(df),
        "Unique_ICB_Codes": df["ICB_Code"].nunique(),
        "Unique_ICB_Names": df["ICB_Name"].nunique(),
        "Missing_ICB_Codes": df["ICB_Code"].isna().sum(),
        "Missing_ICB_Names": df["ICB_Name"].isna().sum(),
        "Duplicate_ICB_Codes": df["ICB_Code"].duplicated().sum()
    })

coverage_summary = pd.DataFrame(coverage_summary)

display(coverage_summary)

,Indicator,Rows,Unique_ICB_Codes,Unique_ICB_Names,Missing_ICB_Codes,Missing_ICB_Names,Duplicate_ICB_Codes
0,CVD mortality,42,42,42,0,0,0
1,Cardiology RTT,36,36,36,0,0,0
2,Coronary heart disease prevalence,42,42,42,0,0,0
3,Heart failure prevalence,42,42,42,0,0,0
4,Hypertension prevalence,42,42,42,0,0,0


#### Create cardiovascular master dataset

In [37]:
cvd_master = (
    cvd_mortality
    .merge(
        cardiology_rtt,
        on=["ICB_Name","ICB_Code"],
        how="left"
    )
    .merge(
        coronary_heart_disease,
        on=["ICB_Name","ICB_Code"],
        how="left"
    )
    .merge(
        heart_failure,
        on=["ICB_Name","ICB_Code"],
        how="left"
    )
    .merge(
        hypertension,
        on=["ICB_Name","ICB_Code"],
        how="left"
    )
)

In [38]:
cvd_master.head()

,ICB_Name,ICB_Code,Under75_CVD_Mortality,Cardiology_RTT_18plus,Coronary_Heart_Disease_Prevalence,Heart_Failure_PrevalencePercentage,Hypertension_Prevalence
0,Cambridgeshire And Peterborough ICB,QUE,52.7,NaN,3.9,1.0,26.1
1,"Buckinghamshire, Oxfordshire And Berkshire Wes...",QU9,54.3,NaN,3.7,1.0,25.9
2,Hertfordshire And West Essex ICB,QM7,56.9,NaN,3.8,0.9,26.0
3,South West London ICB,QWE,57.0,35.81,3.2,0.7,22.5
4,"Bath And North East Somerset, Swindon And Wilt...",QOX,58.2,31.82,4.3,1.3,27.2


In [39]:
cvd_master.to_excel(
    r"..\outputs\CVD_Master.xlsx",
    index=False
)

print("CVD master saved")

CVD master saved


## Respiratory Master Dataset

#### Load respiratory indicators

In [40]:
respiratory_mortality = load_mhs_indicator(
    r"..\Inputs\Under75_Respiratory_Mortality.xlsx",
    "Under75_Respiratory_Mortality"
)

respiratory_rtt = load_mhs_indicator(
    r"..\Inputs\Respiratory_RTT_18plus.xlsx",
    "Respiratory_RTT_18plus"
)

COPD = load_mhs_indicator(
    r"..\Inputs\COPD_Prevalence.xlsx",
    "COPD_Prevalence"
)

asthma = load_mhs_indicator(
    r"..\Inputs\Asthma_Prevalence.xlsx",
    "Asthma_Prevalence"
)

##### Check every raw indicator immediately after loading

In [41]:
indicator_datasets = {
    "Respiratory mortality": respiratory_mortality,
    "Respiratory RTT": respiratory_rtt,
    "COPD prevalence": COPD,
    "Asthma prevalence": asthma
}

coverage_summary = []

for indicator_name, df in indicator_datasets.items():

    coverage_summary.append({
        "Indicator": indicator_name,
        "Rows": len(df),
        "Unique_ICB_Codes": df["ICB_Code"].nunique(),
        "Unique_ICB_Names": df["ICB_Name"].nunique(),
        "Missing_ICB_Codes": df["ICB_Code"].isna().sum(),
        "Missing_ICB_Names": df["ICB_Name"].isna().sum(),
        "Duplicate_ICB_Codes": df["ICB_Code"].duplicated().sum()
    })

coverage_summary = pd.DataFrame(coverage_summary)

display(coverage_summary)

,Indicator,Rows,Unique_ICB_Codes,Unique_ICB_Names,Missing_ICB_Codes,Missing_ICB_Names,Duplicate_ICB_Codes
0,Respiratory mortality,39,39,39,0,0,0
1,Respiratory RTT,36,36,36,0,0,0
2,COPD prevalence,42,42,42,0,0,0
3,Asthma prevalence,42,42,42,0,0,0


#### Create respiratory master dataset

In [42]:
respiratory_master = (
    respiratory_mortality

    .merge(
        respiratory_rtt,
        on=["ICB_Name","ICB_Code"],
        how="left"
    )

    .merge(
        COPD,
        on=["ICB_Name","ICB_Code"],
        how="left"
    )

    .merge(
        asthma,
        on=["ICB_Name","ICB_Code"],
        how="left"
    )

)

In [43]:
respiratory_master.head()

,ICB_Name,ICB_Code,Under75_Respiratory_Mortality,Respiratory_RTT_18plus,COPD_Prevalence,Asthma_Prevalence
0,South West London ICB,QWE,9,70.1,1.1,5.0
1,North East London ICB,QMF,10,79.2,1.0,4.8
2,"Buckinghamshire, Oxfordshire And Berkshire Wes...",QU9,11,NaN,1.3,6.3
3,South East London ICB,QKK,11,74.8,1.2,5.0
4,Hertfordshire And West Essex ICB,QM7,12,NaN,1.5,6.2


In [44]:
respiratory_master.to_excel(
    r"..\outputs\Respiratory_Master.xlsx",
    index=False
)

print("Respiratory master saved")

Respiratory master saved


## Mental Health Master Dataset - to be completed

In [45]:
framework["Status"] = framework["Specialty"].map({
    "Cardiovascular": "Included in POC",
    "Respiratory": "Included in POC",
    "Mental Health": "Future development"
})

## Data Quality Validation

In [46]:
def validate_specialty_master(df, specialty_name):
    print(f"{specialty_name.upper()} DATA QUALITY")
    print("-" * 60)

    print(f"Rows: {len(df)}")
    print(f"Unique ICB codes: {df['ICB_Code'].nunique()}")
    print(
        "Duplicate ICB codes: "
        f"{df['ICB_Code'].duplicated().sum()}"
    )

    print("\nMissing values by column:")
    display(
        df.isna()
        .sum()
        .to_frame("Missing_Values")
    )

    print("\nData types:")
    display(
        df.dtypes
        .astype(str)
        .to_frame("Data_Type")
    )

In [47]:
validate_specialty_master(
    cvd_master,
    "Cardiovascular"
)

validate_specialty_master(
    respiratory_master,
    "Respiratory"
)

CARDIOVASCULAR DATA QUALITY
------------------------------------------------------------
Rows: 42
Unique ICB codes: 42
Duplicate ICB codes: 0

Missing values by column:


,Missing_Values
ICB_Name,0
ICB_Code,0
Under75_CVD_Mortality,0
Cardiology_RTT_18plus,12
Coronary_Heart_Disease_Prevalence,0
Heart_Failure_PrevalencePercentage,0
Hypertension_Prevalence,0



Data types:


,Data_Type
ICB_Name,str
ICB_Code,str
Under75_CVD_Mortality,float64
Cardiology_RTT_18plus,float64
Coronary_Heart_Disease_Prevalence,float64
Heart_Failure_PrevalencePercentage,float64
Hypertension_Prevalence,float64


RESPIRATORY DATA QUALITY
------------------------------------------------------------
Rows: 39
Unique ICB codes: 39
Duplicate ICB codes: 0

Missing values by column:


,Missing_Values
ICB_Name,0
ICB_Code,0
Under75_Respiratory_Mortality,0
Respiratory_RTT_18plus,9
COPD_Prevalence,0
Asthma_Prevalence,0



Data types:


,Data_Type
ICB_Name,str
ICB_Code,str
Under75_Respiratory_Mortality,int64
Respiratory_RTT_18plus,float64
COPD_Prevalence,float64
Asthma_Prevalence,float64


## Create Merged Dataset for all Specialties

In [48]:
specialty_master = pd.merge(
    cvd_master,
    respiratory_master,
    on=["ICB_Name", "ICB_Code"],
    how="outer",
    validate="one_to_one"
)

print("Rows:", len(specialty_master))
print(
    "Unique ICB codes:",
    specialty_master["ICB_Code"].nunique()
)

display(specialty_master.head())

Rows: 42
Unique ICB codes: 42


,ICB_Name,ICB_Code,Under75_CVD_Mortality,Cardiology_RTT_18plus,Coronary_Heart_Disease_Prevalence,Heart_Failure_PrevalencePercentage,Hypertension_Prevalence,Under75_Respiratory_Mortality,Respiratory_RTT_18plus,COPD_Prevalence,Asthma_Prevalence
0,"Bath And North East Somerset, Swindon And Wilt...",QOX,58.2,31.82,4.3,1.3,27.2,15.0,70.7,1.8,7.3
1,"Bedfordshire, Luton And Milton Keynes ICB",QHG,65.6,NaN,3.7,0.9,24.9,13.0,NaN,1.6,6.4
2,Birmingham And Solihull ICB,QHL,84.2,10.91,4.9,0.8,24.4,19.0,68.0,1.5,6.3
3,Black Country ICB,QUA,96.7,37.30,5.5,1.1,25.7,24.0,86.7,2.1,6.6
4,"Bristol, North Somerset And South Gloucestersh...",QUY,62.0,37.85,4.2,1.1,24.8,15.0,80.5,1.9,6.7


## Exploratory Data Analysis

In [49]:
import seaborn as sns

In [50]:
specialty_master.describe()

,Under75_CVD_Mortality,Cardiology_RTT_18plus,Coronary_Heart_Disease_Prevalence,Heart_Failure_PrevalencePercentage,Hypertension_Prevalence,Under75_Respiratory_Mortality,Respiratory_RTT_18plus,COPD_Prevalence,Asthma_Prevalence
count,42.000000,30.000000,42.000000,42.000000,42.000000,39.000000,30.00000,39.000000,39.000000
mean,69.659524,29.238667,4.640476,1.147619,26.769048,18.410256,74.76000,1.979487,6.838462
std,11.498633,9.079489,0.780887,0.295691,2.622742,4.924258,7.63827,0.450835,0.785934
min,52.700000,9.690000,3.200000,0.600000,20.200000,9.000000,55.70000,1.000000,4.800000
25%,60.100000,23.392500,3.950000,1.000000,25.550000,15.000000,70.15000,1.650000,6.400000
50%,65.600000,30.760000,4.700000,1.100000,27.050000,19.000000,75.05000,2.000000,7.000000
75%,78.700000,35.800000,5.200000,1.375000,28.650000,22.000000,80.17500,2.300000,7.450000
max,96.700000,44.190000,6.000000,1.800000,30.900000,29.000000,86.70000,2.800000,8.100000


In [51]:
indicator_columns = [
    column for column in specialty_master.columns
    if column not in ["ICB_Name", "ICB_Code"]
]

indicator_summary = (
    specialty_master[indicator_columns]
    .describe()
    .T
)

indicator_summary["Median"] = (
    specialty_master[indicator_columns]
    .median()
)

indicator_summary["Missing"] = (
    specialty_master[indicator_columns]
    .isna()
    .sum()
)

indicator_summary["Range"] = (
    indicator_summary["max"]
    - indicator_summary["min"]
)

indicator_summary["Coefficient_of_Variation"] = (
    indicator_summary["std"]
    / indicator_summary["mean"]
)

indicator_summary = indicator_summary[
    [
        "count",
        "Missing",
        "mean",
        "Median",
        "std",
        "min",
        "max",
        "Range",
        "Coefficient_of_Variation"
    ]
]

display(
    indicator_summary.round(3)
)

,count,Missing,mean,Median,std,min,max,Range,Coefficient_of_Variation
Under75_CVD_Mortality,42.0,0,69.660,65.60,11.499,52.70,96.70,44.0,0.165
Cardiology_RTT_18plus,30.0,12,29.239,30.76,9.079,9.69,44.19,34.5,0.311
Coronary_Heart_Disease_Prevalence,42.0,0,4.640,4.70,0.781,3.20,6.00,2.8,0.168
Heart_Failure_PrevalencePercentage,42.0,0,1.148,1.10,0.296,0.60,1.80,1.2,0.258
Hypertension_Prevalence,42.0,0,26.769,27.05,2.623,20.20,30.90,10.7,0.098
Under75_Respiratory_Mortality,39.0,3,18.410,19.00,4.924,9.00,29.00,20.0,0.267
Respiratory_RTT_18plus,30.0,12,74.760,75.05,7.638,55.70,86.70,31.0,0.102
COPD_Prevalence,39.0,3,1.979,2.00,0.451,1.00,2.80,1.8,0.228
Asthma_Prevalence,39.0,3,6.838,7.00,0.786,4.80,8.10,3.3,0.115
